# DistilBERT for Five-Class IT Support Classification (Real Data)

**Author:** Vishwajeet Adkine

This notebook fine-tunes `distilbert-base-uncased` to route IT-support questions into **five support queues**, using **real, human-written questions** from Stack Exchange (not synthetic or LLM-generated text).

| Class | Source community | Think of it as the queue for |
|---|---|---|
| Linux | Ask Ubuntu | Linux desktop / OS issues |
| Apple | Ask Different | macOS / iOS issues |
| Android | Android Enthusiasts | Android phone issues |
| Server & Network | Server Fault | sysadmin, servers, networking |
| Security | Information Security | security incidents and hardening |

> **Why this dataset?** Real labelled IT *tickets* are almost never published (privacy), and the public "IT ticket" datasets are synthetic. Real support questions from Stack Exchange are the closest public substitute, but the **label is the community the question was posted in, not a hand-annotated ticket category.** Treat this as a realistic proxy for routing, and say so in any write-up.

**Pipeline:** load data → clean/dedupe → stratified split → TF-IDF baseline → fine-tune DistilBERT → evaluate (macro-F1, confusion matrix, errors) → inference demo → save model.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

## 1. Install and imports

In [ ]:
!pip install -q -U transformers datasets accelerate scikit-learn

In [ ]:
import re, random, json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer, set_seed)

SEED = 42
set_seed(SEED); random.seed(SEED); np.random.seed(SEED)

# ---------------- configuration (edit here) ----------------
DATASET_ID = "flax-sentence-embeddings/stackexchange_titlebody_best_voted_answer_jsonl"
CLASSES = {                       # Stack Exchange community -> class name
    "askubuntu":   "Linux",
    "apple":       "Apple",
    "android":     "Android",
    "serverfault": "Server & Network",
    "security":    "Security",
}
PER_CLASS  = 4000                 # examples per class after cleaning
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN    = 256
EPOCHS     = 3
BATCH_SIZE = 32
LR         = 2e-5
# -----------------------------------------------------------

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 2. Load real data

Each community is a separate split of the dataset. We **stream** it and take a shuffled sample, so Colab doesn't have to download millions of rows. The `title_body` field is the question title plus body (plain text).

If loading fails, see **Troubleshooting** in the README.

In [ ]:
def load_site(site, n, seed=SEED):
    last_err = None
    for extra in ({}, {"trust_remote_code": True}):
        try:
            ds = load_dataset(DATASET_ID, split=site, streaming=True, **extra)
            ds = ds.shuffle(seed=seed, buffer_size=20_000).take(n)
            return [row["title_body"] for row in ds]
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Could not load split '{site}'. Last error: {last_err!r}")

def clean(text):
    text = re.sub(r"\s+", " ", text).strip()
    return text

rows = []
for site, name in CLASSES.items():
    raw = load_site(site, int(PER_CLASS * 1.4))          # oversample, then clean and trim
    seen, kept = set(), []
    for t in raw:
        t = clean(t)
        key = t.lower()
        if len(t) < 40 or key in seen:                    # drop tiny and duplicate questions
            continue
        seen.add(key); kept.append(t)
    kept = kept[:PER_CLASS]
    rows += [{"text": t, "label_name": name} for t in kept]
    print(f"{site:12s} -> {name:17s} {len(kept)} questions")

df = pd.DataFrame(rows)
label_names = sorted(df["label_name"].unique())
label2id = {n: i for i, n in enumerate(label_names)}
id2label = {i: n for n, i in label2id.items()}
df["label"] = df["label_name"].map(label2id)
print("\nTotal:", len(df), "| classes:", label_names)

In [ ]:
df["n_words"] = df["text"].str.split().str.len()
print(df.groupby("label_name")["n_words"].describe()[["mean", "50%", "max"]].round(0))

for name in label_names:
    print(f"\n[{name}]")
    for t in df[df.label_name == name].sample(2, random_state=SEED)["text"]:
        print("  -", t[:220] + ("..." if len(t) > 220 else ""))

## 3. Stratified train / validation / test split (80 / 10 / 10)

In [ ]:
train_df, tmp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
val_df, test_df  = train_test_split(tmp_df, test_size=0.5, stratify=tmp_df["label"], random_state=SEED)
print(len(train_df), len(val_df), len(test_df))

## 4. Baseline: TF-IDF + Logistic Regression

A strong simple baseline keeps the transformer result honest. The second part checks **how much the task depends on obvious brand words** ("Ubuntu", "iPhone", "Android"...) by masking them.

In [ ]:
def run_baseline(train_text, test_text):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=200_000)
    Xtr = vec.fit_transform(train_text)
    clf = LogisticRegression(max_iter=2000, C=5.0)
    clf.fit(Xtr, train_df["label"])
    pred = clf.predict(vec.transform(test_text))
    return accuracy_score(test_df["label"], pred), f1_score(test_df["label"], pred, average="macro")

acc, f1 = run_baseline(train_df["text"], test_df["text"])
print(f"TF-IDF + LR  | accuracy {acc:.4f} | macro-F1 {f1:.4f}")

BRANDS = r"\b(ubuntu|linux|debian|mint|macos|mac os|os x|macbook|mac|iphone|ipad|ios|apple|android|samsung|pixel|nexus)\b"
mask = lambda s: re.sub(BRANDS, " ", s, flags=re.I)
acc_m, f1_m = run_baseline(train_df["text"].map(mask), test_df["text"].map(mask))
print(f"Brand words masked | accuracy {acc_m:.4f} | macro-F1 {f1_m:.4f}")

## 5. Tokenise

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(frame):
    return Dataset.from_pandas(frame[["text", "label"]].reset_index(drop=True), preserve_index=False)

ds = DatasetDict(train=to_ds(train_df), validation=to_ds(val_df), test=to_ds(test_df))
tok = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN),
             batched=True, remove_columns=["text"])
print(tok)

## 6. Fine-tune DistilBERT

Settings: learning rate 2e-5, 3 epochs, batch size 32, 10% warm-up, weight decay 0.01, mixed precision on GPU. The best epoch by validation macro-F1 is kept.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_names), id2label=id2label, label2id=label2id)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro")}

args = TrainingArguments(
    output_dir="distilbert-it-support",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=tok["train"], eval_dataset=tok["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
logs = pd.DataFrame(trainer.state.log_history)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
tr = logs.dropna(subset=["loss"]); ev = logs.dropna(subset=["eval_loss"])
ax[0].plot(tr["step"], tr["loss"], label="train loss")
ax[0].plot(ev["step"], ev["eval_loss"], "o-", label="validation loss")
ax[0].set(title="Loss", xlabel="Step"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ev["epoch"], ev["eval_macro_f1"], "o-", label="macro-F1")
ax[1].plot(ev["epoch"], ev["eval_accuracy"], "o-", label="accuracy")
ax[1].set(title="Validation metrics", xlabel="Epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 7. Evaluate on the held-out test set

In [ ]:
out = trainer.predict(tok["test"])
logits, y_true = out.predictions, out.label_ids
y_pred = logits.argmax(-1)

print(f"Test accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Test macro-F1: {f1_score(y_true, y_pred, average='macro'):.4f}\n")
print(classification_report(y_true, y_pred, target_names=label_names, digits=3))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6.2, 5.2))
im = ax.imshow(cm, cmap="Blues")
ax.set(xticks=range(len(label_names)), yticks=range(len(label_names)),
       xticklabels=label_names, yticklabels=label_names,
       xlabel="Predicted", ylabel="True", title="Confusion matrix (test set)")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
for i in range(len(label_names)):
    for j in range(len(label_names)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im); plt.tight_layout(); plt.show()

## 8. Error analysis

In [ ]:
probs = np.exp(logits - logits.max(-1, keepdims=True)); probs /= probs.sum(-1, keepdims=True)
test_texts = test_df["text"].tolist()          # same order as tok["test"]
wrong = np.where(y_pred != y_true)[0]
print(f"{len(wrong)} wrong out of {len(y_true)}\n")

# most confident mistakes are the most informative
order = wrong[np.argsort(-probs[wrong].max(1))][:8]
for k in order:
    print(f"true={id2label[y_true[k]]:17s} pred={id2label[y_pred[k]]:17s} conf={probs[k].max():.2f}")
    print("   ", test_texts[k][:260].replace("\n", " "), "\n")

## 9. Try it on your own tickets

In [ ]:
from transformers import pipeline
clf = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer,
               top_k=None, device=0 if torch.cuda.is_available() else -1)

samples = [
    "My laptop won't connect to the office VPN since I updated to the latest macOS.",
    "Apt update fails with a GPG error on my fresh install and I can't install anything.",
    "Our nginx reverse proxy returns 502 after we moved the backend to a new subnet.",
    "I got a phishing email and one user already entered their password. What should we do?",
    "Phone battery drains from 100% to 20% in two hours after the last system update.",
]
for s in samples:
    res = sorted(clf(s, truncation=True, max_length=MAX_LEN)[0], key=lambda r: -r["score"])
    print(s)
    print("   ->", ", ".join(f"{r['label']} {r['score']:.2f}" for r in res[:3]), "\n")

## 10. Save the model

In [ ]:
SAVE_DIR = "distilbert-it-support-final"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
json.dump({"classes": CLASSES, "per_class": PER_CLASS, "max_len": MAX_LEN, "epochs": EPOCHS,
           "lr": LR, "batch_size": BATCH_SIZE, "dataset": DATASET_ID},
          open(f"{SAVE_DIR}/training_config.json", "w"), indent=2)

!zip -qr {SAVE_DIR}.zip {SAVE_DIR}
# from google.colab import files; files.download(f"{SAVE_DIR}.zip")      # download
# from google.colab import drive; drive.mount('/content/drive')           # or copy to Drive
# trainer.push_to_hub()                                                    # or publish on Hugging Face
print("Saved to", SAVE_DIR)

## Summary and next steps

- Fine-tuned DistilBERT on real Stack Exchange IT-support questions, five classes, with a TF-IDF baseline, error analysis and an inference demo.
- **Limitation:** labels are communities, not true ticket categories; some questions legitimately fit several classes.
- **Ideas:** add more classes (e.g. Windows via `superuser`), compare `bert-base`/`roberta-base`, add a confidence threshold with a "needs human triage" fallback, test on real tickets from your own organisation.